# Pipelines that survive failure

Idempotent batches, checkpoints, retries with back-off, quarantine, atomic writes, cached results and settings checked up front, so a run that fails halfway can be resumed and gives the same answer.

The lesson's page: https://monstrous.com.ng/learn/data-engineering/series-1/10-pipelines-that-survive-failure/

In [ ]:
# Install what this lesson needs (in Colab, or any fresh environment)
%pip install -q rdkit pandas pydantic pydantic-settings tenacity joblib

import pathlib
import urllib.request
if not pathlib.Path("chemlib.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/David-Alobo/monstrous-learn/main/data-engineering/series-1/chemlib.py", "chemlib.py")

A run over a large library takes hours. Somewhere in those hours something goes wrong: the laptop
goes to sleep, the shared server restarts, a network drive stops answering for ten seconds, or one
record in two million holds a SMILES that RDKit cannot read. A pipeline written as one long loop
loses everything when that happens, and the usual fix, "run it again from the start", is slow and
often wrong: rows already written are written a second time, and a file that was half-written when
the run died is read later as if it were complete.

Industry pipelines are written on the assumption that they *will* fail, and are built so that a
failure costs little: the work is cut into batches, each batch can be re-run without harm, finished
batches are recorded, short-lived errors are retried, bad records are set aside with a reason, and
files are only ever seen whole. This module builds each of those habits by hand, on a library of
12,000 molecules with faults in it, and shows that a run that crashes halfway and is then resumed
gives exactly the same tables as a run that never crashed.

By the end of this module you will be able to:

- check a run's settings before any work starts, with pydantic-settings;
- tell transient failures (retry them) from permanent ones (quarantine them);
- retry with exponential back-off and jitter, by hand and with tenacity;
- write batches idempotently, so a batch that runs twice leaves the same rows as one that runs once;
- record finished batches in a checkpoint table and resume a crashed run from the last good batch;
- explain at-least-once and exactly-once processing, and why idempotency is the practical answer;
- write files atomically, so a crash never leaves a half-written file;
- cache an expensive step on disk with `joblib.Memory`, and know when the cache goes stale.

Everything here runs on any laptop: you need Python, RDKit, pandas, Pydantic, pydantic-settings,
tenacity and joblib. The database is SQLite, which comes with Python. We start with the imports and
a folder for everything the lesson writes:

In [ ]:
import math
import os
import random
import sqlite3
from pathlib import Path

import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

from chemlib import add_defects, library

RDLogger.DisableLog("rdApp.*")  # bad molecules are reported in a table, not as log messages
Path("work").mkdir(exist_ok=True)

## 1. What "surviving failure" means

Failures come in two kinds, and a pipeline has to treat them differently:

- **Transient failures** go away on their own: a timeout, a database that is busy for a moment, a
  web service that answers "try again later". The right response is to wait and try again.
- **Permanent failures** do not: a SMILES with a ring that never closes will fail on every attempt,
  today and tomorrow. Retrying wastes time. The right response is to set the record aside with a
  reason, and carry on with the rest.

And then there are crashes, where the whole process dies (power, memory, a restart). Nothing inside
the program can catch those, so the program has to be written so that starting it again is safe and
cheap. The figure shows the run we build in this module: it crashes at batch 7 and the second run
picks up from there, without redoing batches 0 to 6 and without duplicating batch 7.

![Two runs over thirteen batches. The first run completes and checkpoints batches 0 to 6, saves the rows of batch 7 and crashes before checkpointing it. The second run skips batches 0 to 6, runs batch 7 again and finishes batches 8 to 12; the final tables equal those of a run that never crashed](https://raw.githubusercontent.com/David-Alobo/monstrous-learn/main/data-engineering/series-1/10-pipelines-that-survive-failure/figures/crash-resume.svg)

## 2. Settings checked before a run starts

The cheapest failure is the one that happens before any work is done. A run that reads a batch size
of 0 from a configuration file, or a folder name with a typing mistake, should stop in the first
second with a clear message, not after three hours. **pydantic-settings** does this: a settings
class lists every setting with its type and its limits, reads values from the environment (or a
`.env` file), and refuses to start if any is wrong.

In [ ]:
from pydantic import Field, ValidationError, field_validator, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class RunSettings(BaseSettings):
    """Everything a run needs to know, checked before it starts.

    Each setting can be overridden by an environment variable: CHEMRUN_BATCH_SIZE, and so on.
    """

    model_config = SettingsConfigDict(env_prefix="CHEMRUN_")

    n_molecules: int = Field(12_000, gt=0)
    batch_size: int = Field(1_000, gt=0)
    work_dir: Path = Path("work")
    max_attempts: int = Field(5, ge=1, le=10)
    first_delay_s: float = Field(0.5, gt=0)
    max_delay_s: float = Field(30.0, gt=0)

    @field_validator("work_dir")
    @classmethod
    def folder_exists(cls, folder: Path) -> Path:
        """The output folder must exist before the run, so a typing mistake fails now."""
        if not folder.is_dir():
            raise ValueError(f"no folder called '{folder}'")
        return folder

    @model_validator(mode="after")
    def delays_in_order(self) -> "RunSettings":
        """The first retry delay cannot be longer than the longest delay allowed."""
        if self.first_delay_s > self.max_delay_s:
            raise ValueError("first_delay_s is longer than max_delay_s")
        return self


settings = RunSettings()
for name, value in settings.model_dump().items():
    print(f"{name:14} {value}")

n_molecules    12000
batch_size     1000
work_dir       work
max_attempts   5
first_delay_s  0.5
max_delay_s    30.0


Now the same run started with two mistakes in its environment: a batch size of 0 and the output
folder misspelt. Both are reported together, with the setting, the reason and the value given:

In [ ]:
os.environ["CHEMRUN_BATCH_SIZE"] = "0"
os.environ["CHEMRUN_WORK_DIR"] = "wrok"
try:
    RunSettings()
except ValidationError as err:
    print(f"refused to start: {err.error_count()} bad settings")
    for problem in err.errors():
        print(f"  {problem['loc'][0]}: {problem['msg']} (got {problem['input']!r})")
finally:
    del os.environ["CHEMRUN_BATCH_SIZE"], os.environ["CHEMRUN_WORK_DIR"]

refused to start: 2 bad settings
  batch_size: Input should be greater than 0 (got '0')
  work_dir: Value error, no folder called 'wrok' (got 'wrok')


Without the check, a batch size of 0 would either loop for ever or divide by zero somewhere deep in
the run, and the misspelt folder would be created silently and filled with results nobody finds.
The same class serves every way a run is started (a laptop, a server, a scheduler), because they
all pass settings through the environment.

## 3. One molecule at a time: quarantine, don't crash

Our pipeline does three things to each record: a light validation (can the SMILES be read, is it
one molecule, are its elements ones we expect), canonicalisation, and three descriptors. Module 6
validates properly and Module 7 standardises properly; here the point is what happens when a
record fails. Each problem raises `BadMolecule` with a reason a person can read:

In [ ]:
ALLOWED_ELEMENTS = {"C", "N", "O", "S", "F", "Cl", "Br"}


class BadMolecule(Exception):
    """A permanent problem with a record: trying again will not fix it."""


def prepare(smiles: str) -> dict:
    """Check a SMILES lightly, canonicalise it and compute three descriptors.

    Args:
        smiles: The record's SMILES.

    Returns:
        The canonical SMILES, molecular weight, logP and TPSA.

    Raises:
        BadMolecule: If the record cannot be used, with the reason.
    """
    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        raise BadMolecule("cannot be parsed")
    try:
        Chem.SanitizeMol(mol)
    except Chem.MolSanitizeException as err:
        raise BadMolecule(f"fails sanitisation ({type(err).__name__})") from err
    if len(Chem.GetMolFrags(mol)) > 1:
        raise BadMolecule("more than one component")
    unexpected = sorted({atom.GetSymbol() for atom in mol.GetAtoms()} - ALLOWED_ELEMENTS)
    if unexpected:
        raise BadMolecule(f"unexpected element {', '.join(unexpected)}")
    return {
        "canonical": Chem.MolToSmiles(mol),
        "mol_wt": round(Descriptors.MolWt(mol), 3),
        "logp": round(Descriptors.MolLogP(mol), 3),
        "tpsa": round(Descriptors.TPSA(mol), 2),
    }


def prepare_batch(batch: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Prepare every record of a batch; bad records go to quarantine and the batch carries on.

    Args:
        batch: Records with id and smiles.

    Returns:
        The prepared molecules, and the quarantined records with the reason for each.
    """
    good, bad = [], []
    for record in batch.itertuples(index=False):
        try:
            good.append({"id": record.id, "smiles": record.smiles, **prepare(record.smiles)})
        except BadMolecule as err:
            bad.append({"id": record.id, "smiles": record.smiles, "reason": str(err)})
    good_columns = ["id", "smiles", "canonical", "mol_wt", "logp", "tpsa"]
    return (pd.DataFrame(good, columns=good_columns),
            pd.DataFrame(bad, columns=["id", "smiles", "reason"]))

The library is 12,000 molecules from chemlib, with faults added at a rate of 3%: salts, solvents,
unreadable SMILES, impossible valences, selenium, and duplicates. Sorting by `id` matters more than
it looks: it fixes which records belong to which batch, so batch 7 holds the same records in every
run. Without a fixed order, "resume from batch 7" would mean something different each time.

In [ ]:
source = add_defects(library(settings.n_molecules, seed=10), rate=0.03, seed=11)
source = source.sort_values("id").reset_index(drop=True)
n_batches = math.ceil(len(source) / settings.batch_size)
print(f"{len(source):,} records in {n_batches} batches of up to {settings.batch_size:,}")
print(source["defect"].replace("", "(none)").value_counts().to_string())

12,060 records in 13 batches of up to 1,000
defect
(none)       11700
solvent         60
duplicate       60
valence         60
salt            60
invalid         60
element         60


The 60 duplicates are new rows, so there are 12,060 records and 13 batches, the last one short.
Here is the first batch:

In [ ]:
good, bad = prepare_batch(source.iloc[: settings.batch_size])
print(f"prepared {len(good)}, quarantined {len(bad)}")
print(bad["reason"].value_counts().to_string())

prepared 969, quarantined 31
reason
unexpected element Se                        9
more than one component                      8
fails sanitisation (AtomValenceException)    8
cannot be parsed                             6


Thirty-one bad records in a thousand did not stop the batch. Each one is kept, with its reason, in a quarantine
table where someone can look at it later; nothing is dropped silently. If quarantine starts to fill
quickly (a new supplier's file in a different format, say), that is a signal to stop and look, and
Module 14 turns such counts into alerts.

## 4. Retries with back-off and jitter

Now the transient kind. Suppose the descriptors came from a shared service on another machine,
which sometimes answers "busy". We stand one in with a class that fails its first few calls and then
works, so the lesson behaves the same on every run:

In [ ]:
class TransientError(Exception):
    """A failure that may go away if we wait and try again: a timeout, a busy server."""


class FlakyService:
    """A stand-in for a remote service that fails its first `failures` calls, then works."""

    def __init__(self, failures: int):
        self.failures = failures
        self.calls = 0

    def __call__(self, smiles: str) -> dict:
        """Prepare a molecule, unless this is one of the calls that fail."""
        self.calls += 1
        if self.calls <= self.failures:
            raise TransientError(f"call {self.calls}: service busy")
        return prepare(smiles)

The standard response is **exponential back-off**: wait a little after the first failure, twice as
long after the second, four times as long after the third, up to a ceiling. Waiting longer each
time gives a struggling service room to recover. **Jitter** picks the actual wait at random between
zero and that ceiling ("full jitter"). Without it, a hundred workers that failed together all retry
at the same instants and knock the service over again; with it, their retries spread out. The
function takes `sleep` as an argument so the lesson can record the waits instead of sitting through
them, and `rng` so the random waits are the same on every run:

In [ ]:
def with_retries(func, *args, max_attempts=5, first_delay=0.5, max_delay=30.0,
                 rng=random.Random(), sleep=None):
    """Call func(*args), retrying transient failures with exponential back-off and full jitter.

    Args:
        func: What to call.
        *args: Its arguments.
        max_attempts: How many calls in all before giving up.
        first_delay: The ceiling of the first wait, in seconds; it doubles after each failure.
        max_delay: The largest ceiling, in seconds.
        rng: The random number generator for the jitter.
        sleep: How to wait (time.sleep in real use).

    Returns:
        What func returns.

    Raises:
        TransientError: If every attempt failed. Any other error is raised at once, unretried.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            return func(*args)
        except TransientError as err:
            if attempt == max_attempts:
                raise
            ceiling = min(max_delay, first_delay * 2 ** (attempt - 1))
            delay = rng.uniform(0, ceiling)
            print(f"  attempt {attempt} failed ({err}); waiting {delay:.2f} s of up to {ceiling} s")
            sleep(delay)


waits = []
service = FlakyService(failures=3)
result = with_retries(service, "CC(=O)Oc1ccccc1C(=O)O", rng=random.Random(1), sleep=waits.append)
print(f"succeeded on call {service.calls}: aspirin, MW {result['mol_wt']}")
print(f"waited {sum(waits):.2f} s in all")

  attempt 1 failed (call 1: service busy); waiting 0.07 s of up to 0.5 s
  attempt 2 failed (call 2: service busy); waiting 0.85 s of up to 1.0 s
  attempt 3 failed (call 3: service busy); waiting 1.53 s of up to 2.0 s
succeeded on call 4: aspirin, MW 180.159
waited 2.44 s in all


Two other cases matter as much. A service that never recovers must not be retried for ever: after
`max_attempts` the error is raised, and the run (or the batch) fails visibly. And a permanent error
is not retried at all: a broken SMILES raises `BadMolecule` on the first call, which goes straight
to quarantine:

In [ ]:
try:
    with_retries(FlakyService(failures=10), "CCO", rng=random.Random(1), sleep=waits.append)
except TransientError as err:
    print(f"gave up: {err}")

broken = FlakyService(failures=0)
try:
    with_retries(broken, "C1CC", rng=random.Random(1), sleep=waits.append)
except BadMolecule as err:
    print(f"not retried ({broken.calls} call): {err}")

  attempt 1 failed (call 1: service busy); waiting 0.07 s of up to 0.5 s
  attempt 2 failed (call 2: service busy); waiting 0.85 s of up to 1.0 s
  attempt 3 failed (call 3: service busy); waiting 1.53 s of up to 2.0 s
  attempt 4 failed (call 4: service busy); waiting 1.02 s of up to 4.0 s
gave up: call 5: service busy
not retried (1 call): cannot be parsed


![Back-off delays for twenty clients retrying the same failed service: the ceiling doubles from 0.5 s after each attempt up to 30 s, and each client's actual wait is drawn at random below it, so the retries spread out instead of arriving together](https://raw.githubusercontent.com/David-Alobo/monstrous-learn/main/data-engineering/series-1/10-pipelines-that-survive-failure/figures/backoff.svg)

### The same with tenacity

Writing retry loops by hand in every project leads to small differences and small bugs. The
industry-standard Python library is **tenacity**: a decorator states which errors to retry, when to
stop and how long to wait. `wait_random_exponential` is the same full-jitter back-off as above, and
the limits come from the settings checked in section 2:

In [ ]:
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_random_exponential

service = FlakyService(failures=2)
waits = []


@retry(
    retry=retry_if_exception_type(TransientError),  # BadMolecule is not retried
    stop=stop_after_attempt(settings.max_attempts),
    wait=wait_random_exponential(multiplier=settings.first_delay_s, max=settings.max_delay_s),
    sleep=waits.append,  # leave this out in real use: tenacity then waits with time.sleep
    reraise=True,  # after the last attempt, raise the original error, not a wrapper
)
def prepare_remotely(smiles: str) -> dict:
    """Prepare a molecule on the (flaky) remote service, retrying transient failures."""
    return service(smiles)


random.seed(1)  # tenacity draws its jitter from Python's random module
print(prepare_remotely("CN1C=NC2=C1C(=O)N(C(=O)N2C)C"))
print(f"{service.calls} calls; waits of {[round(w, 2) for w in waits]} s")

{'canonical': 'Cn1c(=O)c2c(ncn2C)n(C)c1=O', 'mol_wt': 194.194, 'logp': -1.029, 'tpsa': 61.82}
3 calls; waits of [0.07, 0.85] s


Which errors count as transient is a decision you write down once: in practice
`TimeoutError`, `ConnectionError`, `sqlite3.OperationalError("database is locked")` and HTTP
responses 429 and 5xx are transient; a bad SMILES, a missing column or HTTP 404 are not. Module 11
uses tenacity in exactly this way against the ChEMBL and PubChem web services.

## 5. Idempotent batches

A step is **idempotent** if running it twice leaves the same result as running it once. That is
the property that makes "just run it again" safe. The common way to write results is not
idempotent: appending rows. Here is the first batch appended twice with pandas, as happens when a
run dies after writing a batch and the batch is then run again:

In [ ]:
naive = sqlite3.connect(Path("work") / "naive.sqlite")
naive.execute("DROP TABLE IF EXISTS molecules")
for attempt in (1, 2):
    good.to_sql("molecules", naive, if_exists="append", index=False)
    count = naive.execute("SELECT COUNT(*), COUNT(DISTINCT id) FROM molecules").fetchone()
    print(f"after writing batch 0 {attempt}x: {count[0]} rows, {count[1]} different molecules")
naive.close()

after writing batch 0 1x: 969 rows, 969 different molecules
after writing batch 0 2x: 1938 rows, 969 different molecules


Every molecule of the batch is now in the table twice, and every count, average and export made
from it is wrong without any error being raised.

The idempotent version has two parts. The table has a **primary key**, the record's `id`, so the
database itself refuses two rows for one molecule. And rows are written with an **upsert**
(`INSERT … ON CONFLICT … DO UPDATE`): insert the row if it is new, overwrite it if it is already
there. SQLite, PostgreSQL and DuckDB all use this syntax. The batch's rows go in inside one
transaction (`with con:`), so a crash halfway through a batch leaves none of its rows, not some:

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS molecules (
    id TEXT PRIMARY KEY, smiles TEXT, canonical TEXT, mol_wt REAL, logp REAL, tpsa REAL);
CREATE TABLE IF NOT EXISTS quarantine (id TEXT PRIMARY KEY, smiles TEXT, reason TEXT);
CREATE TABLE IF NOT EXISTS checkpoints (batch INTEGER PRIMARY KEY, n_good INTEGER, n_bad INTEGER);
"""
UPSERT_MOLECULE = """
INSERT INTO molecules VALUES (:id, :smiles, :canonical, :mol_wt, :logp, :tpsa)
ON CONFLICT (id) DO UPDATE SET smiles = excluded.smiles, canonical = excluded.canonical,
    mol_wt = excluded.mol_wt, logp = excluded.logp, tpsa = excluded.tpsa
"""
UPSERT_QUARANTINE = """
INSERT INTO quarantine VALUES (:id, :smiles, :reason)
ON CONFLICT (id) DO UPDATE SET smiles = excluded.smiles, reason = excluded.reason
"""


def connect(path: Path) -> sqlite3.Connection:
    """Open the run's database, creating its tables if they are not there yet."""
    con = sqlite3.connect(path)
    con.executescript(SCHEMA)
    return con


def save_batch(con: sqlite3.Connection, good: pd.DataFrame, bad: pd.DataFrame) -> None:
    """Write a batch's results idempotently: all its rows in one transaction, upserted by id."""
    with con:  # commits at the end, or rolls everything back if anything fails
        con.executemany(UPSERT_MOLECULE, good.to_dict("records"))
        con.executemany(UPSERT_QUARANTINE, bad.to_dict("records"))


def count_rows(con: sqlite3.Connection) -> str:
    """How many rows each table holds, as one line."""
    tables = ["molecules", "quarantine", "checkpoints"]
    counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables]
    return ", ".join(f"{t} {n:,}" for t, n in zip(tables, counts, strict=True))


demo_path = Path("work") / "demo.sqlite"
demo_path.unlink(missing_ok=True)
con = connect(demo_path)
for attempt in (1, 2):
    save_batch(con, good, bad)
    print(f"after writing batch 0 {attempt}x: {count_rows(con)}")
con.close()

after writing batch 0 1x: molecules 969, quarantine 31, checkpoints 0
after writing batch 0 2x: molecules 969, quarantine 31, checkpoints 0


Idempotency also needs the step itself to be repeatable: the same batch must give the same rows.
That is why the batches have a fixed order, and why the rows carry no timestamp of when they were
computed (a run's start time belongs in a separate table about runs, not in every row).

## 6. Checkpoints, crashes and resuming

A **checkpoint table** records which batches are finished. The pipeline reads it at the start,
skips the finished batches, and writes a checkpoint after each batch's rows are saved. To show a
crash, `run_pipeline` can raise `SimulatedCrash` at a chosen batch, at the worst possible moment:
after the batch's rows are committed but before its checkpoint is written.

In [ ]:
class SimulatedCrash(Exception):
    """Stands in for the process dying: power cut, out of memory, a server restart."""


def run_pipeline(source: pd.DataFrame, db_path: Path, batch_size: int,
                 prepare_fn=prepare_batch, crash_at: int | None = None) -> list[int]:
    """Prepare every batch not yet checkpointed, saving each idempotently.

    Args:
        source: The records, in a fixed order (it decides which records are in which batch).
        db_path: The run's SQLite database.
        batch_size: Records per batch.
        prepare_fn: The function that prepares one batch.
        crash_at: The batch at which to simulate a crash, after saving it and before its checkpoint.

    Returns:
        The batches this call ran.
    """
    con = connect(db_path)
    done = {row[0] for row in con.execute("SELECT batch FROM checkpoints")}
    ran = []
    try:
        for k in range(math.ceil(len(source) / batch_size)):
            if k in done:
                continue
            good, bad = prepare_fn(source.iloc[k * batch_size : (k + 1) * batch_size])
            save_batch(con, good, bad)
            if k == crash_at:
                raise SimulatedCrash(f"died after saving batch {k}, before its checkpoint")
            with con:
                con.execute("INSERT OR REPLACE INTO checkpoints VALUES (?, ?, ?)",
                            (k, len(good), len(bad)))
            ran.append(k)
    finally:
        con.close()
    return ran

First the reference: a run that is never interrupted.

In [ ]:
reference_path = settings.work_dir / "reference.sqlite"
reference_path.unlink(missing_ok=True)
ran = run_pipeline(source, reference_path, settings.batch_size)
print(f"ran batches {ran[0]} to {ran[-1]}")
with sqlite3.connect(reference_path) as con:
    print(count_rows(con))

ran batches 0 to 12
molecules 11,761, quarantine 299, checkpoints 13


Now the same run on a fresh database, crashing at batch 7. The lesson catches the crash so we can
look at what it left behind; a real crash would simply end the process.

In [ ]:
run_path = settings.work_dir / "run.sqlite"
run_path.unlink(missing_ok=True)
try:
    run_pipeline(source, run_path, settings.batch_size, crash_at=7)
except SimulatedCrash as err:
    print(f"crashed: {err}")
with sqlite3.connect(run_path) as con:
    print(count_rows(con))
    finished = [row[0] for row in con.execute("SELECT batch FROM checkpoints ORDER BY batch")]
print(f"checkpointed batches: {finished}")

crashed: died after saving batch 7, before its checkpoint
molecules 7,792, quarantine 208, checkpoints 7
checkpointed batches: [0, 1, 2, 3, 4, 5, 6]


Batches 0 to 6 are checkpointed. Batch 7's rows are in the tables, but its checkpoint is not, so
the database cannot tell that batch 7 is finished. Resuming is simply running the same command
again:

In [ ]:
ran = run_pipeline(source, run_path, settings.batch_size)
print(f"resumed: skipped {len(finished)} finished batches, ran batches {ran}")
with sqlite3.connect(run_path) as con:
    print(count_rows(con))

resumed: skipped 7 finished batches, ran batches [7, 8, 9, 10, 11, 12]
molecules 11,761, quarantine 299, checkpoints 13


Batch 7 ran twice, once before the crash and once after, and the tables hold it once, because
`save_batch` is idempotent. The real test is whether the result equals the uninterrupted run, row for
row:

In [ ]:
def read_tables(db_path: Path) -> dict[str, pd.DataFrame]:
    """Every table of a run's database, sorted by its key, for comparison."""
    keys = {"molecules": "id", "quarantine": "id", "checkpoints": "batch"}
    with sqlite3.connect(db_path) as con:
        return {t: pd.read_sql(f"SELECT * FROM {t} ORDER BY {k}", con) for t, k in keys.items()}


reference, resumed = read_tables(reference_path), read_tables(run_path)
for table in reference:
    pd.testing.assert_frame_equal(reference[table], resumed[table])  # raises if they differ
    print(f"{table}: identical ({len(reference[table]):,} rows)")

molecules: identical (11,761 rows)
quarantine: identical (299 rows)
checkpoints: identical (13 rows)


The quarantine table says why each record was set aside. Joined back to chemlib's record of which
faults it added, it shows how each kind of fault was caught. The 60 duplicates passed: they are
valid molecules, and finding them is Module 7's job:

In [ ]:
caught = source.merge(reference["quarantine"][["id", "reason"]], on="id", how="left")
caught["reason"] = caught["reason"].fillna("(prepared)")
print(pd.crosstab(caught["defect"].replace("", "(none)"), caught["reason"]).to_string())

reason     (prepared)  cannot be parsed  fails sanitisation (AtomValenceException)  more than one component  unexpected element Se
defect                                                                                                                            
(none)          11700                 0                                          0                        0                      0
duplicate          60                 0                                          0                        0                      0
element             1                 0                                          0                        0                     59
invalid             0                60                                          0                        0                      0
salt                0                 0                                          0                       60                      0
solvent             0                 0                                          0 

One record marked `element` was prepared. Looking at it (`c1cc(c8ccccc8)ccc1-c2nc(c8ccncc8)c(O)o2`)
explains why: chemlib adds selenium by replacing an aliphatic carbon, and this molecule has none, so
the "fault" left it unchanged and it is a valid molecule. Labels of what is wrong can be wrong too;
checking a pipeline against them is how such cases come to light.

### At-least-once and exactly-once

Every batch here is processed **at least once**: a batch that crashed before its checkpoint is run
again, so some batches are processed twice. What we would like is **exactly once**: every batch's
effect appears once, no matter what fails. Within one database, a transaction can give that: if the
rows and the checkpoint were written in the same transaction, they would be saved together or not
at all. But the moment a step touches two systems (rows into a database and a file on disk, or a
call to a web service and a row recording it), no transaction covers both, and there is always a
moment where one has happened and the other has not. That is exactly the moment our simulated crash
picked.

The practical answer, used across the industry, is **at-least-once delivery plus idempotent
steps**: accept that work may be repeated, and make repeating it harmless. The effect is the same as
exactly once, and it needs nothing more than a key and an upsert. Kleppmann's book (further
reading) explains this in depth for databases and message queues.

At scale the pattern is the same. For two million molecules, batches of 10,000 give 200
checkpoints; a crash costs at most one batch of work. Smaller batches lose less in a crash but write
more checkpoints; most pipelines choose batches that take seconds to minutes. With PostgreSQL
(Module 4) the tables and the upserts are the same, and many workers can write to one database at
once (Module 9).

## 7. Atomic writes

Results usually leave the database as files: a CSV for a colleague, a Parquet file for the next
pipeline. Writing a file takes time, and a crash in the middle leaves a **half-written file** that
looks like a complete one. Here is the export done the obvious way, crashing a third of the way
through:

In [ ]:
import csv

molecules = reference["molecules"]
export_path = settings.work_dir / "molecules.csv"


def write_rows(handle, table: pd.DataFrame, crash_after: int | None = None) -> None:
    """Write a table as CSV to an open file, one row at a time, optionally crashing partway."""
    writer = csv.writer(handle)
    writer.writerow(table.columns)
    for i, row in enumerate(table.itertuples(index=False)):
        if i == crash_after:
            raise SimulatedCrash(f"died after writing {i:,} rows")
        writer.writerow(row)


export_path.unlink(missing_ok=True)
try:
    with open(export_path, "w", newline="", encoding="utf-8") as handle:
        write_rows(handle, molecules, crash_after=4_000)
except SimulatedCrash as err:
    print(f"crashed: {err}")
print(f"the file exists: {export_path.exists()}, rows a reader sees: {len(pd.read_csv(export_path)):,}")

crashed: died after writing 4,000 rows
the file exists: True, rows a reader sees: 4,000


A reader the next morning finds a well-formed CSV with fewer rows and no sign that anything went
wrong. The fix is an **atomic write**: write to a temporary file in the same folder, make sure it
is on disk (`flush` and `os.fsync`), then rename it over the target with `os.replace`. Renaming a
file within one folder is atomic on Windows, macOS and Linux: other programs see either the old file
or the new one, never a mixture. If the run dies first, the target is untouched and the temporary
file is removed:

In [ ]:
def write_csv_atomic(table: pd.DataFrame, path: Path, crash_after: int | None = None) -> None:
    """Write a table as CSV so that readers only ever see the old file or the complete new one.

    Args:
        table: What to write.
        path: The file to write.
        crash_after: Simulate a crash after this many rows (for the lesson).
    """
    tmp = path.with_name(path.name + ".tmp")
    try:
        with open(tmp, "w", newline="", encoding="utf-8") as handle:
            write_rows(handle, table, crash_after)
            handle.flush()
            os.fsync(handle.fileno())  # the bytes are on the disk, not only in memory
        os.replace(tmp, path)  # atomic: the old file or the new one, never half of one
    finally:
        tmp.unlink(missing_ok=True)


write_csv_atomic(molecules, export_path)
print(f"complete export: {len(pd.read_csv(export_path)):,} rows")
try:
    write_csv_atomic(molecules, export_path, crash_after=4_000)
except SimulatedCrash as err:
    print(f"crashed: {err}")
print(f"after the crash: {len(pd.read_csv(export_path)):,} rows, "
      f"temporary file left behind: {export_path.with_name('molecules.csv.tmp').exists()}")

complete export: 11,761 rows
crashed: died after writing 4,000 rows
after the crash: 11,761 rows, temporary file left behind: False


The same applies to every file a pipeline writes: Parquet files, JSON reports, model files. Two
details matter. The temporary file must be in the same folder (a rename between two drives is a
copy, which is not atomic). And readers should ignore `*.tmp` files. Object stores such as Amazon
S3 give the same guarantee differently: an upload becomes visible only once it is complete.

## 8. Cached results with joblib.Memory

Resuming skips finished batches, but sometimes you rerun everything on purpose: a new database, a
new export format, a colleague's copy of the pipeline. If a step is expensive (descriptors here;
3D conformers, docking or quantum chemistry in real work), computing it again for unchanged input
is waste. `joblib.Memory` remembers a function's results on disk, keyed by a hash of its arguments:
call it again with the same arguments, in the same run or a later one, and it returns the stored
result without running the function. The list `computed` records each time the function body really
runs. (`memory.cache` is usually written as a decorator, `@memory.cache`; here it is applied after
setting a short name, for the reason in the comment.)

In [ ]:
from joblib import Memory

memory = Memory(settings.work_dir / "cache", verbose=0)
memory.clear(warn=False)  # start the lesson with an empty cache
computed = []


def prepare_batch_cached(batch: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """prepare_batch, with its results kept on disk: each different batch is computed once."""
    computed.append(batch["id"].iloc[0])
    return prepare_batch(batch)


# joblib names the cache folder after where the function is defined: for a script, its full path,
# which can pass Windows' 260-character path limit. A short module name keeps the folder short.
prepare_batch_cached.__module__ = "module10"
prepare_batch_cached = memory.cache(prepare_batch_cached)


for name in ("first", "second"):
    path = settings.work_dir / f"cached-{name}.sqlite"
    path.unlink(missing_ok=True)
    before = len(computed)
    run_pipeline(source, path, settings.batch_size, prepare_fn=prepare_batch_cached)
    print(f"{name} full run: {len(computed) - before} batches computed")
same = read_tables(settings.work_dir / "cached-second.sqlite")["molecules"]
print("cached results equal the reference:", same.equals(reference["molecules"]))

first full run: 13 batches computed
second full run: 0 batches computed
cached results equal the reference: True


The cache lives in `work/cache`, so a new Python session, or tomorrow's run, reuses it too. One
warning, which catches out many users: joblib notices when the code of the cached function changes,
but **not** when a function it calls changes. If we changed `prepare` (a new descriptor, a fixed
bug), `prepare_batch_cached` would keep returning the old results. Clear the cache when the logic
changes (`memory.clear()`), or pass a version number as an argument so new logic gets new cache
entries. A cache that can return stale chemistry is worse than no cache.

## 9. The same ideas in industry tools

Everything in this module is what workflow tools do for you at scale. Prefect, Dagster and Airflow
let you declare retries with back-off on each task, cache task results by their inputs, and record
which tasks of a run finished so a failed run resumes from the failure. dlt, a loading library,
keeps the state of each load and merges rows by key. Series 2 swaps such tools in (dlt, dbt and
Prefect); the ideas you need to judge them are the ones built here:

- check the settings first, and fail before any work is done;
- retry what is transient, with back-off and jitter and a limit; quarantine what is permanent;
- cut the work into batches with a fixed order, write them idempotently, and checkpoint them;
- write files atomically;
- cache expensive steps, and know what makes the cache stale.

## Try it yourself

The homework for this module is in `homework.md`: a few questions answered by running code like the
code above. Answers are published after the cohort's deadline.

## Further reading

- Kleppmann, M. (2017). *Designing Data-Intensive Applications: The Big Ideas Behind Reliable,
  Scalable, and Maintainable Systems*. O'Reilly Media. ISBN 978-1-449-37332-0. Chapters 7 (transactions) and 11 (stream processing: idempotence and
  exactly-once).
- Beyer, B., Jones, C., Petoff, J. & Murphy, N. R. (eds.) (2016). *Site Reliability Engineering*.
  O'Reilly Media. Chapter 22, Addressing Cascading Failures (retries, back-off and jitter).
  <https://sre.google/sre-book/addressing-cascading-failures/>
- Brooker, M. (2015). Exponential backoff and jitter. *AWS Architecture Blog*, 4 March 2015.
  <https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/>
- The tenacity documentation. <https://tenacity.readthedocs.io/en/latest/>
- The pydantic-settings documentation. *Settings Management*.
  <https://docs.pydantic.dev/latest/concepts/pydantic_settings/>
- The joblib documentation. *Disk-based caching* (the Memory class).
  <https://joblib.readthedocs.io/en/stable/user_guide/memory.html>
- The SQLite documentation. *UPSERT*. <https://www.sqlite.org/lang_upsert.html>